# Toy Monte Carlo — Optional self-study

**Analysis Techniques in Neutrino Astronomy · PhD School**

This notebook is an **optional extension** of the main HESE hands-on. It uses a generated Monte Carlo sample and an intentionally simplified detector. No external dataset is needed, and **the toy results are not IceCube predictions**.

**Learning goals:** reweight a generated sample to different spectra, understand selection efficiency and effective area, and explore energy and angular resolution.

Work through the exercises in order. The *Bonus* tasks are not required.

## A. Generate and reweight Monte Carlo events

The generated energies follow $q(E)\propto E^{-1}$. This is convenient because the events are approximately uniform in equal bins of $\log_{10}E$.

To represent a physical spectral shape proportional to $E^{-\gamma}$ with the **same** generated events, we assign relative weights:

$$
w_\gamma(E) = \left(\frac{E}{E_{\mathrm{pivot}}}\right)^{1-\gamma}.
$$

We use $E_{\mathrm{pivot}}=10\,\mathrm{TeV}$. The weights have **arbitrary normalization**: they are not physical expected event counts.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.figsize':(8,4.8),'axes.grid':True,'grid.alpha':.23})
rng = np.random.default_rng(2026)
N_GENERATED = 160_000
E_MIN, E_MAX, E_PIVOT = 1e2, 1e7, 1e4  # GeV
GAMMAS = [2.0, 2.7, 3.8]
# Inverse-CDF sampling for q(E) \propto E^-1 over finite limits.
E = 10**rng.uniform(np.log10(E_MIN), np.log10(E_MAX), N_GENERATED)
log_bins = np.logspace(np.log10(E_MIN),np.log10(E_MAX),30)

def relative_weight(energy, gamma):
    """Relative to q(E) ∝ E^-1; pivot-normalized weight (arbitrary units)."""
    return (energy/E_PIVOT)**(1-gamma)

print('Generated MC:',len(E),' / E range:',(E.min(),E.max()))

### Exercise 1 — Reweighting (lecture slide 24)

1. Plot the **unweighted** event counts in equal logarithmic energy bins. Are they approximately flat?
2. On the same axes, plot weighted counts for $\gamma=2.0$, $2.7$, and $3.8$. Use logarithmic axes.
3. Which spectrum gives relatively more weight to high-energy events? Why do the curves meet near the pivot energy?

**Hint:** use `np.histogram(E, bins=log_bins, weights=...)` and the function `relative_weight`.

In [ ]:
# Exercise 1: histogram the raw and reweighted energies.
# TODO: make the comparison plot.

### Exercise 2 — Relative event yields

1. For each $\gamma$, calculate the sum of event weights **below** and **above** $E_{\mathrm{pivot}}$. Compare the results.
2. Explain why these weighted sums are **not** physical predicted event counts.

**Hint:** a physical rate needs more than a relative spectral shape; think about flux normalization and exposure.

In [ ]:
# Exercise 2: compare weighted sums above and below E_PIVOT.
# TODO: calculate and print the totals for each spectral index.

## B. A simple toy detector

We now simulate a detector selection, a reconstructed deposited energy, and reconstructed event directions. Because this toy keeps **all generated events**, including those not selected, we can calculate the selection efficiency directly.

The model uses deliberately simplified assumptions about acceptance, visible energy, topology and angular smearing. Treat the resulting plots as illustrations only.

In [ ]:
# Toy acceptance with a smooth energy turn-on + weaker horizon sensitivity.
A_GENERATION_CM2 = 1e8  # arbitrary reference area, 10,000 m^2
coszen = rng.uniform(-1,1,len(E))
phi_true = rng.uniform(0,2*np.pi,len(E))
p_energy = .002 + .28/(1+(3e4/E)**1.2)
p_accept = p_energy*(.68 + .32*(1-coszen)/2)
selected = rng.uniform(size=len(E)) < p_accept

# Toy reco morphology: 0=cascade, 1=track, correlated with the true interaction label.
interaction = rng.choice([1,2],size=len(E),p=[.7,.3])  # 1=CC, 2=NC
topology_true = np.where(interaction==2, 0, rng.choice([0,1],len(E),p=[.55,.45]))
p_reco_track = np.where(topology_true==1,.91,.08)
topology_reco = (rng.uniform(size=len(E))<p_reco_track).astype(int)

# Reconstructed deposited energy includes random visible-energy fraction and smearing.
f_visible = np.where(interaction==1, .78, .35)
f_visible = np.where(topology_true==1, f_visible*.66, f_visible)
e_reco = E*f_visible*10**rng.normal(0,.18,size=len(E))

# Unit sphere: both zenith and azimuth of true/reconstructed directions available in toy.
th_true = np.arccos(coszen)
n_true=np.column_stack([np.sin(th_true)*np.cos(phi_true),
                        np.sin(th_true)*np.sin(phi_true), np.cos(th_true)])
e1=np.column_stack([-np.sin(phi_true),np.cos(phi_true),np.zeros(len(E))])
e2=np.cross(n_true,e1)
sigma_deg=np.where(topology_true==1,1.0,8.0)*(E/1e5)**(-.10)
sigma_rad=np.deg2rad(np.clip(sigma_deg,.25,20.0))
err1=rng.normal(size=len(E))*sigma_rad
err2=rng.normal(size=len(E))*sigma_rad
n_reco=n_true + err1[:,None]*e1 + err2[:,None]*e2
n_reco/=np.linalg.norm(n_reco,axis=1)[:,None]

print(f'Selected toy events: {selected.sum():,} / {len(E):,} ({selected.mean():.2%})')

### Exercise 3 — Selection efficiency and effective area

For energy bin $i$, define

$$
\epsilon_i = \frac{N_{\mathrm{selected},i}}{N_{\mathrm{generated},i}},
\qquad A_{\mathrm{eff},i} = A_{\mathrm{generation}}\,\epsilon_i.
$$

1. Plot $\epsilon$ as a function of **true energy**.
2. Plot the toy $A_{\mathrm{eff}}$ as a function of true energy. Why does it increase with energy?

**Bonus:** repeat the efficiency plot in two true-zenith regions and compare them.

In [ ]:
# Exercise 3: selection efficiency and toy effective area.
# TODO: histogram E for all events and for selected events, then compute their ratio.

### Exercise 4 — Reconstructed energy and energy resolution

The variable `e_reco` represents the toy **reconstructed deposited energy**, while `E` is the **true neutrino energy**. These are different physical quantities.

1. Make a scatter plot of $E_{\mathrm{reco}}$ versus $E_{\mathrm{true}}$ for selected events. Use logarithmic axes and draw the line $E_{\mathrm{reco}}=E_{\mathrm{true}}$.
2. Plot the distribution of $E_{\mathrm{reco}}/E_{\mathrm{true}}$ for selected events.
3. Is the reconstructed energy typically lower than the true energy? Explain the **bias** (shift) and **resolution** (spread) in the plot.

**Bonus:** compare the energy-ratio distributions for tracks and cascades.

In [ ]:
# Exercise 4: reconstructed-versus-true energy and their ratio.
# TODO: plot e_reco[selected] against E[selected], then histogram their ratio.

### Exercise 5 — Angular resolution

A directional error must include **both zenith and azimuth**. The 3D opening angle between the true and reconstructed unit vectors is

$$
\psi = \arccos\!\left(\hat{n}_{\mathrm{true}}\cdot\hat{n}_{\mathrm{reco}}\right).
$$

1. Calculate $\psi$ (in degrees) for the selected events and plot it versus true energy. Distinguish reconstructed tracks from cascades.
2. Calculate the **50% containment angle** (the median of $\psi$) for each topology. Which has better angular resolution?
3. **Bonus:** repeat the 50% containment calculation in true-energy bins to study its energy dependence.

**Hint:** use `np.sum(n_true * n_reco, axis=1)`, `np.clip`, `np.arccos` and `np.rad2deg`.

In [ ]:
# Exercise 5: full 3D angular separation and containment.
# TODO: compute psi_deg using n_true and n_reco, then compare track/cascade events.

### Bonus — Point-spread function (PSF)

If you want a more advanced challenge, estimate the angular probability density per unit **solid angle**, $dP/d\Omega$, from the selected events.

For an angular annulus between $\psi_{\mathrm{low}}$ and $\psi_{\mathrm{high}}$ (angles in radians), its solid angle is

$$
\Delta\Omega = 2\pi\left(\cos\psi_{\mathrm{low}}-\cos\psi_{\mathrm{high}}\right).
$$

Divide the fraction of events in each annulus by its $\Delta\Omega$ and verify that $\sum_i (dP/d\Omega)_i\,\Delta\Omega_i\approx 1$.

This is a **bonus** because the solid-angle normalization is more technical than the containment-angle calculation.

In [ ]:
# Bonus: estimate dP/dOmega, normalized over solid angle.
# TODO (optional).

## C. Discussion

- Why simulate events with a convenient non-physical spectrum and reweight them later?
- How does detector selection affect the observed energy distribution?
- Why do tracks and cascades have different angular resolutions?

**Take-home:** event weights specify a spectral hypothesis, while selection and reconstruction describe the detector response. All quantities here are illustrative toy-model results.